# Lesson 3: Chatbot Example

In this lesson, you will familiarize yourself with the chatbot example you will work on during this course. The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.

## Import Libraries

In [1]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

## Tool Functions

In [2]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  

In [3]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    
    # Use arxiv to find the papers 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info  
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids

In [5]:
search_papers("intelligence")

Results are saved in: papers/intelligence/papers_info.json


['0712.3825v1', '2110.01835v1', '2204.10358v1', '2103.06769v1', '2110.01831v1']

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.

In [6]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
 
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."

In [7]:
extract_info('0712.3825v1')

'{\n  "title": "Tests of Machine Intelligence",\n  "authors": [\n    "Shane Legg",\n    "Marcus Hutter"\n  ],\n  "summary": "Although the definition and measurement of intelligence is clearly of fundamental importance to the field of artificial intelligence, no general survey of definitions and tests of machine intelligence exists. Indeed few researchers are even aware of alternatives to the Turing test and its many derivatives. In this paper we fill this gap by providing a short survey of the many tests of machine intelligence that have been proposed.",\n  "pdf_url": "https://arxiv.org/pdf/0712.3825v1",\n  "published": "2007-12-22"\n}'

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [8]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping

This code handles tool mapping and execution.

In [9]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
    
    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [10]:
load_dotenv() 
client = anthropic.Anthropic()

### Query Processing

> **⚠️ Deprecation Notice**
> The model `claude-3-7-sonnet-20250219` was deprecated in February 2026 and is no longer supported. All code in this notebook has been updated to use `claude-sonnet-4-6` to ensure everything runs as expected.

In [11]:
def process_query(query):
    
    messages = [{'role': 'user', 'content': query}]
    
    response = client.messages.create(max_tokens = 2024,
                                  #model = 'claude-3-7-sonnet-20250219', #deprecated model
                                  model = 'claude-sonnet-4-6', 
                                  tools = tools,
                                  messages = messages)
    
    process_query = True
    while process_query:
        assistant_content = []

        for content in response.content:
            if content.type == 'text':
                
                print(content.text)
                assistant_content.append(content)
                
                if len(response.content) == 1:
                    process_query = False
            
            elif content.type == 'tool_use':
                
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({"role": "user", 
                                  "content": [
                                      {
                                          "type": "tool_result",
                                          "tool_use_id": tool_id,
                                          "content": result
                                      }
                                  ]
                                })
                response = client.messages.create(max_tokens = 2024,
                                  #model = 'claude-3-7-sonnet-20250219', #deprecated model
                                  model = 'claude-sonnet-4-6', 
                                  tools = tools,
                                  messages = messages) 
                
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query = False

### Chat Loop

In [12]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"

To access the `papers` folder: 1) click on the `File` option on the top menu of the notebook and 2) click on `Open` and then 3) click on `L3`.

In [ ]:
chat_loop()

Type your queries or 'quit' to exit.



Query:  search for intelligence book


Calling tool search_papers with args {'topic': 'intelligence book'}
Results are saved in: papers/intelligence_book/papers_info.json
Now, let me fetch the details of all these papers simultaneously!
Calling tool extract_info with args {'paper_id': '0712.3825v1'}
Calling tool extract_info with args {'paper_id': '2110.01835v1'}

Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.5: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_01LzPybQ2am9yDD4qPSxZrFg. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CZCZ9vsGwh4neXVv4WjHs'}



Query:  search for intelligence


Calling tool search_papers with args {'topic': 'intelligence'}
Results are saved in: papers/intelligence/papers_info.json
I found 5 papers related to **intelligence**. Let me now retrieve detailed information about all of them simultaneously!
Calling tool extract_info with args {'paper_id': '0712.3825v1'}
Calling tool extract_info with args {'paper_id': '2110.01835v1'}

Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.5: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_016L9uVkdckBEbdwbPEL2yfT. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CZCZDApFbBFWyqZWXCSZZ'}


...

It seems like you've just typed "computers" — that's a broad topic! Let me search for the latest papers on arXiv related to computers for you.
Calling tool search_papers with args {'topic': 'computers'}
Results are saved in: papers/computers/papers_info.json
Let me now extract information about all these papers simultaneously!
Calling tool extract_info with args {'paper_id': '1312.3300v1'}
Calling tool extract_info with args {'paper_id': '2207.05241v1'}

Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.5: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_01XNVGAaSW2ygrifre3tLmAY. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CZCZDwaYtvVNHt6BPuR74'}



Query:  hi


Hello! How can I help you today? If you're looking for research papers or information on a specific topic, feel free to ask and I can search for relevant papers on arXiv! 😊





Query:  search for paper on Algebra


I'll search for papers on Algebra right away!
Calling tool search_papers with args {'topic': 'Algebra'}
Results are saved in: papers/algebra/papers_info.json
I found **5 papers** on Algebra. Let me now fetch detailed information for all of them simultaneously!
Calling tool extract_info with args {'paper_id': '2411.11095v3'}
Calling tool extract_info with args {'paper_id': '0905.2613v3'}

Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.5: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_017QCjAefMdnFyWifrhdigBa. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CZCZMNJHUBWaFdukBj91L'}
